# MML Chapter 9: Linear Regression

> _Starting from a simple noise model, the chapter builds from MLE (least squares) to MAP (ridge) to full Bayesian inference — each adding one more ingredient of uncertainty._

This notebook accompanies Chapter 9 of *Mathematics for Machine Learning* (Deisenroth, Faisal, Ong).

Each section runs a hands-on experiment:

1. **MLE (normal equations)** — implement from scratch and verify against `numpy.linalg.lstsq`
2. **Geometric interpretation** — project $y$ onto $\text{col}(X)$ in $\mathbb{R}^3$; visualise the residual
3. **Ridge regression (MAP)** — coefficient shrinkage as $\lambda$ varies; ridge trace
4. **Bayesian linear regression** — posterior $\mathcal{N}(m_N, S_N)$; prior → posterior evolution
5. **Predictive distribution** — epistemic + aleatoric uncertainty; out-of-distribution widening
6. **Feature maps** — sin/cos basis vs polynomial basis on periodic data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats

np.random.seed(42)

C = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Setup complete.')

---
## 1. MLE via Normal Equations (Least Squares)

**Model**: $y_n = x_n^\top \theta + \varepsilon_n$, $\varepsilon_n \sim \mathcal{N}(0, \sigma^2)$.

**MLE** minimises the residual sum of squares:
$$\hat\theta_{\text{ML}} = (X^\top X)^{-1} X^\top y$$

This comes from setting $\partial \|y - X\theta\|^2 / \partial\theta = 0$, which gives the **normal equations** $X^\top X\theta = X^\top y$.

In [ ]:
# ── Generate data from a known linear model ──────────────────────────────────
true_theta = np.array([2.0, -1.5, 0.8])   # intercept, slope1, slope2
N = 50
sigma = 0.5

# Design matrix: [1, x1, x2]
X_raw = np.random.randn(N, 2)
X = np.column_stack([np.ones(N), X_raw])  # add intercept column
y = X @ true_theta + np.random.normal(0, sigma, N)

# ── From-scratch implementation of normal equations ───────────────────────────
def normal_equations(X, y):
    """θ = (XᵀX)⁻¹ Xᵀy — raises LinAlgError if XᵀX is singular."""
    XtX = X.T @ X
    Xty = X.T @ y
    return np.linalg.solve(XtX, Xty)  # solve is numerically preferable to explicit inverse

theta_ne   = normal_equations(X, y)
theta_np, *_ = np.linalg.lstsq(X, y, rcond=None)   # NumPy reference

print("True θ         :", true_theta)
print("Normal equations:", theta_ne.round(4))
print("numpy.linalg   :", theta_np.round(4))
print("Max difference  :", np.max(np.abs(theta_ne - theta_np)))

# ── Residuals and sigma² MLE ──────────────────────────────────────────────────
y_hat      = X @ theta_ne
residuals  = y - y_hat
sigma2_mle = np.mean(residuals**2)
print(f"\nTrue σ²={sigma**2:.2f}  |  σ²_MLE={sigma2_mle:.4f}  (biased; unbiased={np.var(residuals, ddof=len(true_theta)):.4f})")

# ── Quick diagnostic plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.scatter(y_hat, y, s=25, alpha=0.7)
lim = [min(y.min(), y_hat.min()) - 0.5, max(y.max(), y_hat.max()) + 0.5]
ax.plot(lim, lim, 'k--', lw=1.5, label='Perfect fit')
ax.set_xlabel('ŷ  (predicted)')
ax.set_ylabel('y  (observed)')
ax.set_title('Predicted vs Observed')
ax.legend()

ax2 = axes[1]
ax2.stem(np.arange(N), residuals, markerfmt='o', linefmt='C0-', basefmt='k-')
ax2.axhline(0, color='k', lw=1)
ax2.set_xlabel('Index')
ax2.set_ylabel('Residual y − ŷ')
ax2.set_title('Residuals (should look like white noise)')

plt.tight_layout()
plt.suptitle('Fig 1: MLE via Normal Equations', y=1.02, fontsize=13)
plt.show()

---
## 2. Geometric Interpretation

With $N$ data points, the design matrix $X \in \mathbb{R}^{N \times D}$ defines a $D$-dimensional subspace — its **column space** $\text{col}(X)$.

The MLE prediction $\hat{y} = X(X^\top X)^{-1}X^\top y = P_X y$ is the **orthogonal projection** of $y$ onto $\text{col}(X)$.

The residual $y - \hat{y}$ is **perpendicular** to every column of $X$. That is why the equations are called *normal* equations — the residual is normal (perpendicular) to the column space.

**Experiment**: $N=3$ data points, $D=2$ features (plus intercept). $y \in \mathbb{R}^3$, $\text{col}(X)$ is a 2-D plane in $\mathbb{R}^3$.

In [ ]:
# 3 data points, 2-dimensional design matrix (no intercept for visual clarity)
X3 = np.array([[1.0, 0.0],
               [0.5, 1.0],
               [0.0, 1.0]])
y3 = np.array([1.5, 2.0, 0.8])

# Hat matrix and projection
P = X3 @ np.linalg.solve(X3.T @ X3, X3.T)  # P = X(XᵀX)⁻¹Xᵀ
y_hat3   = P @ y3
residual3 = y3 - y_hat3

print("y         :", y3)
print("ŷ = P·y  :", y_hat3.round(4))
print("residual  :", residual3.round(4))
print()
# Verify orthogonality: X.T @ residual should be ~0
print("Xᵀ(y−ŷ) [should be ≈ 0]:", (X3.T @ residual3).round(10))

# ── 3D plot of the projection ─────────────────────────────────────────────────
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Span the column space: sample a grid in θ-space and map to R^3
t1 = np.linspace(-1, 2.5, 30)
t2 = np.linspace(-1, 2.5, 30)
T1, T2 = np.meshgrid(t1, t2)
# col(X) plane: all vectors of the form θ1·x1 + θ2·x2
plane_x = T1*X3[0,0] + T2*X3[0,1]
plane_y = T1*X3[1,0] + T2*X3[1,1]
plane_z = T1*X3[2,0] + T2*X3[2,1]
ax.plot_surface(plane_x, plane_y, plane_z, alpha=0.25, color=C[0])

# Vectors
origin = np.zeros(3)
for vec, col, lbl in [
    (y3,       'black',  'y  (target)'),
    (y_hat3,   C[0],     'ŷ = Pₓy  (projection)'),
]:
    ax.quiver(*origin, *vec, color=col, arrow_length_ratio=0.12, lw=2, label=lbl)

# Residual vector from ŷ to y
ax.quiver(*y_hat3, *residual3, color='red', arrow_length_ratio=0.15, lw=2, label='y − ŷ  (residual, ⊥ col(X))')

# Column vectors of X
for j, col in enumerate([C[2], C[3]]):
    ax.quiver(*origin, *X3[:,j], color=col, arrow_length_ratio=0.12, lw=1.5,
              linestyle='--', label=f'x_{j+1} (col {j+1} of X)')

ax.set_xlabel('Dim 1')
ax.set_ylabel('Dim 2')
ax.set_zlabel('Dim 3')
ax.set_title('Projection of y onto col(X)\n(blue plane = column space of X)')
ax.legend(fontsize=8, loc='upper left')
plt.suptitle('Fig 2: Geometric Interpretation of Least Squares', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. Ridge Regression (MAP with Gaussian Prior)

**MAP** with prior $p(\theta) = \mathcal{N}(0, b^2 I)$ gives ridge regression:

$$\hat\theta_{\text{MAP}} = \arg\min_\theta \left[\|y - X\theta\|^2 + \lambda\|\theta\|^2\right] = (X^\top X + \lambda I)^{-1}X^\top y$$

where $\lambda = \sigma^2 / b^2$. The matrix $X^\top X + \lambda I$ is always positive definite, so the solution always exists — even when $N < D$.

**Ridge trace**: as $\lambda \to 0$, coefficients approach OLS; as $\lambda \to \infty$, they shrink to 0.

In [ ]:
# Dataset: deliberately correlated features to show ridge stabilisation
N_ridge = 40
true_theta_ridge = np.array([1.0, 2.0, -1.5, 0.5, 1.8])  # 5 features
D_ridge = len(true_theta_ridge)
sigma_ridge = 1.0

# Design matrix with correlated columns
rng = np.random.RandomState(7)
Z = rng.randn(N_ridge, D_ridge)
# Introduce correlation between features 1 and 2
Z[:, 1] = 0.9 * Z[:, 0] + 0.1 * rng.randn(N_ridge)
X_ridge = Z
y_ridge = X_ridge @ true_theta_ridge + rng.normal(0, sigma_ridge, N_ridge)

lambdas = np.logspace(-3, 3, 200)

def ridge_fit(X, y, lam):
    D = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * np.eye(D), X.T @ y)

coeff_paths = np.array([ridge_fit(X_ridge, y_ridge, lam) for lam in lambdas])
ols_coeffs  = np.linalg.lstsq(X_ridge, y_ridge, rcond=None)[0]

# Training MSE for each lambda
train_mse_ridge = []
for lam in lambdas:
    coeffs = ridge_fit(X_ridge, y_ridge, lam)
    train_mse_ridge.append(np.mean((y_ridge - X_ridge @ coeffs)**2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for j in range(D_ridge):
    ax.semilogx(lambdas, coeff_paths[:, j], lw=2, label=f'θ_{j+1} (true={true_theta_ridge[j]})')
ax.axhline(0, color='k', lw=0.8)
for j in range(D_ridge):
    ax.axhline(ols_coeffs[j], color=C[j], linestyle=':', alpha=0.5)
ax.set_xlabel('Regularisation strength  λ  (log scale)')
ax.set_ylabel('Coefficient value')
ax.set_title('Ridge Trace: Coefficient Paths as λ Varies')
ax.legend(fontsize=8)

ax2 = axes[1]
ax2.semilogx(lambdas, train_mse_ridge, lw=2, color=C[0], label='Train MSE')
ax2.axvline(1.0, color='grey', linestyle='--', lw=1.5, label='λ=1 (moderate)')
ax2.set_xlabel('λ  (log scale)')
ax2.set_ylabel('Training MSE')
ax2.set_title('Training MSE vs Regularisation')
ax2.legend()

plt.tight_layout()
plt.suptitle('Fig 3: Ridge Regression — MAP with Gaussian Prior', y=1.02, fontsize=13)
plt.show()

print(f"OLS coefficients :  {ols_coeffs.round(3)}")
print(f"True θ           :  {true_theta_ridge}")
print(f"Ridge λ=1.0      :  {ridge_fit(X_ridge, y_ridge, 1.0).round(3)}")
print(f"Ridge λ=100      :  {ridge_fit(X_ridge, y_ridge, 100).round(3)}  (strongly shrunk)")

---
## 4. Bayesian Linear Regression: Posterior

With prior $p(\theta) = \mathcal{N}(\theta \mid m_0, S_0)$ and likelihood $p(y \mid X, \theta) = \mathcal{N}(y \mid X\theta, \sigma^2 I)$,
the posterior is:

$$p(\theta \mid X, y) = \mathcal{N}(\theta \mid m_N, S_N)$$

$$S_N^{-1} = S_0^{-1} + \sigma^{-2} X^\top X, \qquad m_N = S_N\left(S_0^{-1}m_0 + \sigma^{-2}X^\top y\right)$$

**Intuition**: The posterior precision is the sum of prior precision and data precision. As $N$ grows, the data term dominates and the posterior concentrates near the MLE.

**Experiment**: 1-D regression. Plot prior, posterior after 5 points, posterior after 50 points.

In [ ]:
# 1-D BLR: θ = [intercept, slope]; feature vector φ(x) = [1, x]
true_w = np.array([0.5, 1.5])  # intercept=0.5, slope=1.5
sigma2_blr = 0.25              # observation noise variance

# Prior: broad isotropic Gaussian
m0 = np.zeros(2)
S0 = 4.0 * np.eye(2)          # prior variance 4 in each direction

def design_matrix_1d(x):
    """φ(x) = [1, x] for each point → (N,2) matrix."""
    return np.column_stack([np.ones_like(x), x])

def blr_posterior(X_phi, y, m0, S0, sigma2):
    """Return posterior mean m_N and covariance S_N."""
    S0_inv = np.linalg.inv(S0)
    S_N_inv = S0_inv + (1/sigma2) * X_phi.T @ X_phi
    S_N = np.linalg.inv(S_N_inv)
    m_N = S_N @ (S0_inv @ m0 + (1/sigma2) * X_phi.T @ y)
    return m_N, S_N

# Generate all data at once; reveal progressively
N_total = 50
x_all = np.random.uniform(-3, 3, N_total)
y_all = design_matrix_1d(x_all) @ true_w + np.random.normal(0, np.sqrt(sigma2_blr), N_total)

# Stages
stages = {'Prior (N=0)': (None, None), 'N=5': (5,), 'N=50': (50,)}

# Sample weight vectors from prior / posteriors to show function uncertainty
x_plot = np.linspace(-4, 4, 200)
Phi_plot = design_matrix_1d(x_plot)

n_samples = 6
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

configs = [
    (0,  'Prior (N=0)',       m0, S0),
    (5,  'Posterior (N=5)',   *blr_posterior(design_matrix_1d(x_all[:5]),  y_all[:5],  m0, S0, sigma2_blr)),
    (50, 'Posterior (N=50)',  *blr_posterior(design_matrix_1d(x_all[:50]), y_all[:50], m0, S0, sigma2_blr)),
]

for ax, (n_shown, title, m, S) in zip(axes, configs):
    # Sample function lines from this distribution
    w_samples = np.random.multivariate_normal(m, S, n_samples)
    for ws in w_samples:
        ax.plot(x_plot, Phi_plot @ ws, alpha=0.35, lw=1.2, color=C[0])
    # Mean function
    ax.plot(x_plot, Phi_plot @ m, color=C[0], lw=2.5, label='Posterior mean')
    # True function
    ax.plot(x_plot, Phi_plot @ true_w, 'k--', lw=1.5, label='True function', alpha=0.7)
    # Data shown so far
    if n_shown > 0:
        ax.scatter(x_all[:n_shown], y_all[:n_shown], s=25, color='k', zorder=5, label=f'{n_shown} data pts')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-7, 8)
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.suptitle('Fig 4: Bayesian Linear Regression — Prior → Posterior Evolution', y=1.02, fontsize=13)
plt.show()

m_N5,  S_N5  = blr_posterior(design_matrix_1d(x_all[:5]),  y_all[:5],  m0, S0, sigma2_blr)
m_N50, S_N50 = blr_posterior(design_matrix_1d(x_all[:50]), y_all[:50], m0, S0, sigma2_blr)

print(f"True θ     : {true_w}")
print(f"Posterior mean (N=5) : {m_N5.round(4)}")
print(f"Posterior mean (N=50): {m_N50.round(4)}")
print(f"Posterior std (N=5)  : {np.sqrt(np.diag(S_N5)).round(4)}")
print(f"Posterior std (N=50) : {np.sqrt(np.diag(S_N50)).round(4)}  (much tighter)")

---
## 5. Predictive Distribution

Marginalising the posterior over $\theta$ gives the **predictive distribution** for a new point $x^*$:

$$p(y^* \mid x^*, X, y) = \mathcal{N}\!\left(y^* \;\Big|\; m_N^\top\phi(x^*),\; \underbrace{\phi(x^*)^\top S_N \phi(x^*)}_\text{epistemic} + \underbrace{\sigma^2}_\text{aleatoric}\right)$$

- **Epistemic uncertainty** ($\phi(x^*)^\top S_N \phi(x^*)$): parameter uncertainty; shrinks with more data; grows outside training range.
- **Aleatoric uncertainty** ($\sigma^2$): irreducible observation noise; constant regardless of data size.

In [ ]:
# Use the N=50 posterior from Section 4
m_N, S_N = m_N50, S_N50

x_star = np.linspace(-6, 6, 300)   # extended beyond training range [-3,3]
Phi_star = design_matrix_1d(x_star)

# Predictive mean
pred_mean = Phi_star @ m_N

# Predictive variance = epistemic + aleatoric
epistemic_var = np.array([phi.T @ S_N @ phi for phi in Phi_star])  # φᵀS_Nφ
aleatoric_var = sigma2_blr * np.ones_like(x_star)                   # σ²
total_var     = epistemic_var + aleatoric_var
pred_std      = np.sqrt(total_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: full predictive distribution
ax = axes[0]
ax.fill_between(x_star, pred_mean - 2*pred_std, pred_mean + 2*pred_std,
                alpha=0.2, color=C[0], label='±2σ total')
ax.fill_between(x_star,
                pred_mean - 2*np.sqrt(aleatoric_var),
                pred_mean + 2*np.sqrt(aleatoric_var),
                alpha=0.3, color=C[1], label='±2σ aleatoric only')
ax.plot(x_star, pred_mean, color=C[0], lw=2, label='Predictive mean')
ax.plot(x_star, Phi_star @ true_w, 'k--', lw=1.5, alpha=0.7, label='True function')
ax.scatter(x_all[:50], y_all[:50], s=20, color='k', zorder=5, alpha=0.6, label='Training data')
ax.axvspan(-3, 3, alpha=0.05, color='green', label='Training range')
ax.set_xlim(-6, 6)
ax.set_ylim(-10, 12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Predictive Distribution (N=50)')
ax.legend(fontsize=8)

# Right: uncertainty decomposition
ax2 = axes[1]
ax2.plot(x_star, np.sqrt(epistemic_var), lw=2, label='Epistemic std  √(φᵀS_Nφ)')
ax2.plot(x_star, np.sqrt(aleatoric_var), lw=2, linestyle='--', label=f'Aleatoric std  σ={np.sqrt(sigma2_blr):.2f}')
ax2.plot(x_star, pred_std, lw=2, linestyle=':', label='Total predictive std')
ax2.axvspan(-3, 3, alpha=0.08, color='green', label='Training range')
ax2.set_xlabel('x')
ax2.set_ylabel('Standard deviation')
ax2.set_title('Uncertainty Decomposition')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Fig 5: Predictive Distribution — Epistemic + Aleatoric Uncertainty', y=1.02, fontsize=13)
plt.show()

# Show how uncertainty grows outside training range
in_range  = np.abs(x_star) <= 3
out_range = np.abs(x_star) > 3
print(f"Mean epistemic std  inside  training range : {np.sqrt(epistemic_var[in_range]).mean():.4f}")
print(f"Mean epistemic std  outside training range : {np.sqrt(epistemic_var[out_range]).mean():.4f}")
print(f"Aleatoric std (constant)                   : {np.sqrt(sigma2_blr):.4f}")

---
## 6. Feature Maps: Sinusoidal vs Polynomial Basis

Raw input $x$ may have a non-linear relationship with $y$. By applying a **feature map** $\phi: \mathbb{R} \to \mathbb{R}^K$, we keep the model *linear in $\theta$* while fitting non-linear patterns.

$$f(x; \theta) = \phi(x)^\top \theta, \quad \hat\theta = (\Phi^\top\Phi)^{-1}\Phi^\top y$$

**Experiment**: True data is periodic (noisy sine). Compare:
- **Fourier features**: $\phi(x) = [1, \sin(x), \cos(x), \sin(2x), \cos(2x), \ldots]$ — adapted to the periodicity
- **Polynomial features**: $\phi(x) = [1, x, x^2, \ldots, x^K]$ — not adapted; needs high degree to mimic periodic behaviour

In [ ]:
# True data: periodic
def true_periodic(x):
    return 1.5 * np.sin(x) + 0.5 * np.cos(2*x)

N_feat = 30
noise_feat = 0.3
x_feat = np.sort(np.random.uniform(0, 2*np.pi, N_feat))
y_feat = true_periodic(x_feat) + np.random.normal(0, noise_feat, N_feat)

x_fine_feat = np.linspace(-0.5, 2*np.pi + 0.5, 400)

# ── Feature maps ─────────────────────────────────────────────────────────────
def fourier_features(x, n_harmonics=4):
    """[1, sin(x), cos(x), sin(2x), cos(2x), ..., sin(nh*x), cos(nh*x)]"""
    cols = [np.ones_like(x)]
    for k in range(1, n_harmonics + 1):
        cols += [np.sin(k * x), np.cos(k * x)]
    return np.column_stack(cols)

def poly_features(x, degree):
    return np.vander(x, degree + 1, increasing=True)

# Fit
n_harmonics = 3           # gives 2*3+1 = 7 Fourier features
poly_degree = 9           # polynomial of similar parameter count

Phi_four_train = fourier_features(x_feat, n_harmonics)
Phi_four_test  = fourier_features(x_fine_feat, n_harmonics)
theta_four, *_ = np.linalg.lstsq(Phi_four_train, y_feat, rcond=None)
y_four = Phi_four_test @ theta_four

Phi_poly_train = poly_features(x_feat, poly_degree)
Phi_poly_test  = poly_features(x_fine_feat, poly_degree)
theta_poly, *_ = np.linalg.lstsq(Phi_poly_train, y_feat, rcond=None)
y_poly = Phi_poly_test @ theta_poly

# Train MSE
mse_four = np.mean((y_feat - Phi_four_train @ theta_four)**2)
mse_poly = np.mean((y_feat - Phi_poly_train @ theta_poly)**2)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, y_pred, title, mse, color in [
    (axes[0], y_four, f'Fourier basis (harmonics={n_harmonics}, K={Phi_four_train.shape[1]})', mse_four, C[0]),
    (axes[1], y_poly, f'Polynomial basis (degree={poly_degree}, K={poly_degree+1})', mse_poly, C[1]),
]:
    ax.scatter(x_feat, y_feat, s=30, color='k', zorder=5, label='Data')
    ax.plot(x_fine_feat, true_periodic(x_fine_feat), 'k--', lw=1.5, alpha=0.6, label='True function')
    ax.plot(x_fine_feat, y_pred, lw=2.5, color=color, label='Model fit')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)
    ax.set_ylim(-4, 4)
    ax.text(0.05, 0.95, f'Train MSE = {mse:.4f}', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.axvline(0, color='grey', lw=0.8, alpha=0.5)
    ax.axvline(2*np.pi, color='grey', lw=0.8, alpha=0.5, label='Training range')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.suptitle('Fig 6: Feature Maps — Fourier vs Polynomial Basis for Periodic Data', y=1.02, fontsize=13)
plt.show()

print(f"Fourier ({2*n_harmonics+1} features): Train MSE = {mse_four:.5f}")
print(f"Polynomial (degree {poly_degree}, {poly_degree+1} features): Train MSE = {mse_poly:.5f}")
print()
print("Key observation: Fourier features exploit the known periodicity of the data.")
print("Even with fewer parameters, they generalise better.")
print("The polynomial fit may oscillate wildly outside the training range.")

# Extrapolation check (beyond training range)
x_extrap = np.linspace(-2, 3*np.pi, 200)
y_extrap_four = fourier_features(x_extrap, n_harmonics) @ theta_four
y_extrap_poly = poly_features(x_extrap, poly_degree) @ theta_poly

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.scatter(x_feat, y_feat, s=25, color='k', zorder=5, label='Training data')
ax.plot(x_extrap, true_periodic(x_extrap), 'k--', lw=1.5, alpha=0.6, label='True function')
ax.plot(x_extrap, y_extrap_four, lw=2, color=C[0], label='Fourier extrapolation')
ax.plot(x_extrap, y_extrap_poly, lw=2, color=C[1], linestyle='--', label='Polynomial extrapolation')
ax.axvspan(0, 2*np.pi, alpha=0.07, color='green', label='Training range')
ax.set_ylim(-8, 8)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Extrapolation Beyond Training Range')
ax.legend(fontsize=9)
plt.tight_layout()
plt.suptitle('Fig 6b: Extrapolation Comparison', fontsize=13)
plt.show()

---
## Summary

| Section | Formula | Key insight |
|---------|---------|-------------|
| **MLE (normal equations)** | $\hat\theta = (X^\top X)^{-1}X^\top y$ | MLE under Gaussian noise = least squares |
| **Geometric interpretation** | $\hat y = P_X y$, $P_X = X(X^\top X)^{-1}X^\top$ | Residual $y - \hat y$ is orthogonal to $\text{col}(X)$ |
| **Ridge regression (MAP)** | $\hat\theta = (X^\top X + \lambda I)^{-1}X^\top y$ | Always invertible; MAP with Gaussian prior; shrinks coefficients |
| **Bayesian posterior** | $p(\theta\mid X,y) = \mathcal{N}(m_N, S_N)$ | Posterior precision = prior precision + data precision |
| **Predictive distribution** | $\mathcal{N}(m_N^\top\phi(x^*),\;\phi^\top S_N\phi + \sigma^2)$ | Epistemic uncertainty grows outside training distribution |
| **Feature maps** | $\phi(x) = [\sin(x), \cos(x), \ldots]$ | Inductive bias in feature design dominates model selection |

**Connections**:
- **C8**: MLE for Gaussian $\to$ C9 MLE for regression (Gaussian noise model makes them equivalent)
- **C10 (PCA)**: Same $X^\top X$ eigendecomposition appears in PCA and evidence maximisation
- **C11 (GMMs)**: EM algorithm used in evidence maximisation (§9.6) is the same as EM for GMMs
- **C12 (GPs)**: Bayesian linear regression in the kernel dual is exactly GP regression